# GTA-UAV Multi-Tile Pose Likelihood

## TL;DR

This notebook is the reproducible reader-facing audit for the pre-registered `retrieval top-R × VOP top-k` pilot. It reads immutable JSONL caches and the fitted calibrator; it does not run retrieval, matching, or training. The decision cell below is populated from the experiment summary.

In [1]:
from pathlib import Path
import json, sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

cwd = Path.cwd().resolve()
roots = [cwd, cwd.parent, cwd.parent.parent]
ROOT = next(path for path in roots if (path / 'game4loc').is_dir())
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
from game4loc.evaluate.gta_pose_likelihood import FEATURE_NAMES, PoseLikelihoodCalibrator, feature_vector
from fit_gta_pose_likelihood import candidate_pool, raw_choice

RUN = ROOT / 'work_dir/gta_pose_likelihood_runs/gta_multitile_20260903'
SUMMARY_PATH = RUN / 'pilot_summary.json'
ARTIFACT_PATH = RUN / 'pilot_calibrator.json'
EVAL_CACHE_PATH = RUN / 'pilot_test345.jsonl'
for path in (SUMMARY_PATH, ARTIFACT_PATH, EVAL_CACHE_PATH):
    assert path.exists(), f'Missing required pilot artifact: {path}'
summary = json.loads(SUMMARY_PATH.read_text(encoding='utf-8'))
artifact = json.loads(ARTIFACT_PATH.read_text(encoding='utf-8'))
records = [json.loads(line) for line in EVAL_CACHE_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
calibrator = PoseLikelihoodCalibrator(artifact)
display(Markdown(f"**Recorded decision:** `{summary['decision']}`  \n**Model:** `{summary['calibration']['model_type']}`  \n**Evaluation queries:** {len(records)}"))

**Recorded decision:** `REJECT`  
**Model:** `hist_gradient_boosting`  
**Evaluation queries:** 345

## Context & Methods

Retrieval, the full-teacher Exp C VOP checkpoint, and sparse matcher defaults are frozen. Each query has up to 20 candidates: five retrieved satellite tiles and four VOP angles per tile. A query-grouped 70/15/15 split with seed `20260903` is used for fitting, model selection, and temperature calibration. The target is candidate error below 20 m. The official evaluator receives only inference-time retrieval, VOP, and geometry features; ground truth appears only in this offline audit cache.

In [2]:
forbidden = ('error', 'ground_truth', 'success', 'improves', 'catastrophic')
assert not any(token in name for name in FEATURE_NAMES for token in forbidden)
assert artifact['feature_names'] == list(FEATURE_NAMES)
display(pd.DataFrame({'feature': FEATURE_NAMES, 'inference_only': True}))

,feature,inference_only
0,retrieval_rank,True
1,retrieval_score,True
2,retrieval_gap_top1,True
3,retrieval_margin_next,True
4,vop_angle_rank,True
5,vop_prob,True
6,vop_top_prob,True
7,vop_entropy,True
8,vop_concentration,True
9,log_retained_matches,True


## Data

The table reports actual cached cardinality, class balance, and measured VOP/matcher time. Timing is the sum of per-query candidate-cache measurements and therefore represents exhaustive top-5 × top-4 generation, not adaptive official-evaluator latency.

In [3]:
all_candidates = [candidate for record in records for candidate in record['candidates']]
data_audit = pd.DataFrame([{
    'queries': len(records),
    'candidates': len(all_candidates),
    'mean_candidates/query': len(all_candidates) / len(records),
    'candidate_<20m_rate_%': 100 * np.mean([c['success_20m'] for c in all_candidates]),
    'mean_VOP_time_s/query': np.mean([r['vop_time_s'] for r in records]),
    'mean_match_time_s/query': np.mean([r['match_time_s'] for r in records]),
    'cache_candidate_time_s/query': np.mean([r['vop_time_s'] + r['match_time_s'] for r in records]),
}])
display(data_audit.round(4))

,queries,candidates,mean_candidates/query,candidate_<20m_rate_%,mean_VOP_time_s/query,mean_match_time_s/query,cache_candidate_time_s/query
0,345,6900,20.0,13.913,0.2197,1.5835,1.8032


## Results


In [4]:
rows = []
for key, label in [('coarse_top1', 'Coarse top-1'), ('legacy_top1_vop', 'Legacy top-1 VOP'), ('raw_top5x4', 'Raw top-5×4'), ('oracle_top5x4', 'Oracle top-5×4')]:
    row = summary['pilot'][key]
    rows.append({'variant': label, 'Dis@1_m': row['Dis@1_m'], 'MA@20_%': row['MA@20_pct'], 'worse_%': row['worse_than_coarse_pct'], 'catastrophic_%': row['catastrophic_50m_pct']})
row = summary['adaptive_eval']
rows.append({'variant': 'Calibrated adaptive', 'Dis@1_m': row['Dis@1_m'], 'MA@20_%': row['MA@20_pct'], 'worse_%': row['worse_than_coarse_pct'], 'catastrophic_%': row['catastrophic_50m_pct']})
result_table = pd.DataFrame(rows).set_index('variant')
display(result_table.round(3))
fig, axes = plt.subplots(1, 2, figsize=(11, 3.8))
result_table['Dis@1_m'].plot.bar(ax=axes[0], color='#4c78a8', title='Mean localization error (lower is better)')
result_table['MA@20_%'].plot.bar(ax=axes[1], color='#59a14f', title='MA@20 (higher is better)')
axes[0].set_ylabel('metres'); axes[1].set_ylabel('%'); plt.tight_layout(); plt.show()

,Dis@1_m,MA@20_%,worse_%,catastrophic_%
variant,,,,
Coarse top-1,140.020,7.536,0.000,0.000
Legacy top-1 VOP,86.357,42.609,11.884,2.899
Raw top-5×4,158.978,45.217,16.232,7.826
Oracle top-5×4,26.593,65.507,2.899,0.580
Calibrated adaptive,74.719,47.246,5.507,0.580


/tmp/ipykernel_37842/74398564.py:12: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  axes[0].set_ylabel('metres'); axes[1].set_ylabel('%'); plt.tight_layout(); plt.show()


In [5]:
matrix = np.stack([feature_vector(c) for c in all_candidates])
labels = np.asarray([c['success_20m'] for c in all_candidates], dtype=float)
probabilities = calibrator.predict_proba_matrix(matrix)
order = np.argsort(probabilities)
bins = np.array_split(order, 15)
reliability = pd.DataFrame([{
    'mean_probability': probabilities[idx].mean(),
    'empirical_<20m_rate': labels[idx].mean(),
    'count': len(idx),
} for idx in bins if len(idx)])
metrics = pd.Series(summary['calibration']['metrics'], name='value')
display(metrics.to_frame().round(5))
fig, ax = plt.subplots(figsize=(5, 4))
ax.plot([0, 1], [0, 1], '--', color='0.6', label='ideal')
ax.plot(reliability['mean_probability'], reliability['empirical_<20m_rate'], 'o-', label='15 equal-mass bins')
ax.set(xlabel='Predicted probability', ylabel='Observed <20m frequency', title='Calibration curve', xlim=(0, 1), ylim=(0, 1))
ax.legend(); plt.tight_layout(); plt.show()

,value
positive_rate,0.13913
AUPRC,0.50871
NLL,0.28327
Brier,0.08716
ECE_15_equal_mass,0.02055
AUROC,0.86820


/tmp/ipykernel_37842/2118974597.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.legend(); plt.tight_layout(); plt.show()


In [6]:
cal_rows, raw_rows = [], []
for record in records:
    pool = candidate_pool(record, 5, 4)
    selected, confidence = calibrator.best_candidate(pool)
    raw = raw_choice(pool)
    cal_rows.append((float(confidence), float(selected['error_m'])))
    raw_rows.append((float(raw['inliers']), float(raw['error_m'])))
cal_rows = sorted(cal_rows, key=lambda row: row[0], reverse=True)
raw_rows = sorted(raw_rows, key=lambda row: row[0], reverse=True)
risk_rows = []
for coverage in np.linspace(.1, 1., 10):
    keep = max(1, int(np.ceil(len(records) * coverage)))
    risk_rows.append({'coverage': coverage, 'calibrated': np.mean([r[1] for r in cal_rows[:keep]]), 'raw inlier': np.mean([r[1] for r in raw_rows[:keep]])})
risk = pd.DataFrame(risk_rows)
display(risk.round(3))
ax = risk.plot(x='coverage', y=['calibrated', 'raw inlier'], marker='o', figsize=(6, 4), title='Risk–coverage')
ax.set(xlabel='Coverage', ylabel='Mean error (m)'); plt.tight_layout(); plt.show()

,coverage,calibrated,raw inlier
0,0.1,16.877,29.578
1,0.2,19.282,27.456
2,0.3,19.847,27.603
3,0.4,20.713,27.190
4,0.5,21.269,26.801
5,0.6,22.285,26.672
6,0.7,22.945,26.675
7,0.8,23.529,25.800
8,0.9,26.451,29.594
9,1.0,75.902,158.978


/tmp/ipykernel_37842/3563170012.py:17: UserWarning: FigureCanvasAgg is non-interactive, and thus cannot be shown
  ax.set(xlabel='Coverage', ylabel='Mean error (m)'); plt.tight_layout(); plt.show()


In [7]:
fixed = pd.DataFrame(summary['fixed_configuration_rows'])
display(fixed[['retrieval_topk', 'orientation_topk', 'hypotheses_per_query', 'Dis@1_m', 'MA@20_pct']].round(3))
display(pd.Series(summary['adaptive_policy'], name='value').to_frame())

,retrieval_topk,orientation_topk,hypotheses_per_query,Dis@1_m,MA@20_pct
0,1,2,2,66.867,41.333
1,1,4,4,62.986,43.333
2,3,2,6,78.806,43.667
3,3,4,12,58.528,44.667
4,5,2,10,78.858,44.000
5,5,4,20,59.238,45.000


,value
expansion_stages,"[1, 3, 5]"
orientation_topk,4
threshold_r1,0.358866
threshold_r3,0.432158
abstain_threshold,0.206784
mean_hypotheses,6.826667
metrics,"{'query_count': 300, 'Dis@1_m': 58.07090990884..."


## Takeaways

The final decision is governed by the pre-registered oracle and calibration gates, not by visual preference. `KEEP` permits full same-area cache generation; `REJECT` stops the method line after the one fixed nonlinear follow-up. `NEEDS ONE FOLLOW-UP` is valid only when that follow-up has not yet run.

In [8]:
gates = summary['gates']
audit = pd.DataFrame([
    {'check': 'Oracle headroom gate', 'passed': gates['oracle_headroom_pass']},
    {'check': 'Calibration gate', 'passed': gates['calibration_pass']},
    {'check': 'No GT/error labels in official feature schema', 'passed': True},
    {'check': 'Single HGB follow-up limit respected', 'passed': bool(summary['calibration']['followup_used']) or summary['calibration']['model_type'] == 'logistic'},
])
display(audit)
display(Markdown(f"### Final pilot status: `{summary['decision']}`"))

,check,passed
0,Oracle headroom gate,True
1,Calibration gate,False
2,No GT/error labels in official feature schema,True
3,Single HGB follow-up limit respected,True


### Final pilot status: `REJECT`